# GTEx model with traits

💡 **Environment:** `clamp-analyses`  

## Load libraries

In [45]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)

source(here("config.R"))

## Output directory

In [46]:
output_data_dir <- config$GTEx$DATASET_FOLDER
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

# GTEx model

In [47]:
clamp_gtex_data <- readRDS(here('output/gtex/gtex_CellMarker_2024_CLAMP.rds'))

In [48]:
head(clamp_gtex_data$B)
head(clamp_gtex_data$Z)

,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
LV1,0.003515103,0.12077255,0.03759850,0.111187477,0.17431512,0.16568284,-0.07361237,-0.06406146,0.0340904639,-0.01193462,⋯,0.124919246,0.13089421,0.01755203,0.02647548,0.037244386,-0.004276459,-0.01320797,0.17837765,0.051750005,0.09908475
LV2,-0.152494031,-0.30102684,-0.17996463,-0.127454900,-0.19658856,-0.22645209,-0.11882937,-0.14566668,-0.2531348749,-0.18173210,⋯,-0.178118250,-0.05775840,-0.14254041,-0.20662046,-0.175588078,-0.046581324,-0.24361526,-0.16815216,-0.298404294,-0.28413497
LV3,-0.163522476,-0.09381197,-0.13369426,-0.143002828,0.05078467,-0.13920504,-0.17715985,0.07824148,-0.1172904211,0.03409911,⋯,0.301871777,0.15837912,0.07625206,0.21647397,0.140901009,-0.119342636,0.12048466,0.04556854,-0.009725146,0.00920801
LV4,0.873860320,-0.09400434,0.14629256,0.042776115,0.16375204,0.14647352,0.15383083,-0.11996810,0.2546400598,0.32524062,⋯,0.102808977,0.13052483,0.09883889,0.16830722,0.028216304,0.188722838,-0.30541741,-0.06557762,-0.208605326,0.26854142
LV5,-0.065012557,0.01466201,0.03095714,0.008455027,0.01721777,0.02092374,-0.12396106,0.69664186,-0.0763711649,0.01643529,⋯,0.005451017,-0.02640570,-0.06017774,-0.02496730,0.003097109,-0.127847228,0.67034541,-0.05477048,-0.072706323,-0.04218699
LV6,0.068835455,-0.01101978,0.02555418,-0.113679265,-0.11159923,-0.06737068,-0.02742618,-0.02778976,-0.0005461837,-0.04494157,⋯,-0.124841569,-0.07636996,-0.13284074,-0.09179978,-0.132252113,-0.106935944,-0.04561036,-0.16248933,-0.076002365,-0.05853467


,LV1,LV2,LV3,LV4,LV5,LV6,LV7,LV8,LV9,LV10,⋯,LV403,LV404,LV405,LV406,LV407,LV408,LV409,LV410,LV411,LV412
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WASH7P,0,0.0000000,0,0,0,0.0000000,0,0,0,0.0000000,⋯,0.000000,0.00000,0,0.0000000,0,0.0000000,0.0000000,0,0,0
RP11-34P13.15,0,0.2196016,0,0,0,0.0000000,0,0,0,0.0000000,⋯,0.000000,0.00000,0,0.0000000,0,0.3194992,0.0000000,0,0,0
RP11-34P13.16,0,0.1837671,0,0,0,0.0000000,0,0,0,0.0000000,⋯,0.000000,0.00000,0,0.0000000,0,0.3132951,0.0000000,0,0,0
RP11-34P13.18,0,0.1939171,0,0,0,0.0000000,0,0,0,0.2807416,⋯,0.233273,0.00000,0,0.0000000,0,0.0000000,0.1066575,0,0,0
AP006222.2,0,0.4803007,0,0,0,0.1313486,0,0,0,0.0000000,⋯,0.000000,0.71251,0,0.4230056,0,0.0000000,0.0000000,0,0,0
MTND1P23,0,0.0000000,0,0,0,0.0000000,0,0,0,0.0000000,⋯,0.000000,0.00000,0,0.0000000,0,0.4771836,0.0000000,0,0,0


## Phenoplier traits

In [49]:
gtex_phenoplier <- readRDS(here('data/gtex/phenoplier/gtex-phenoplier.rds'))

gtex_phenoplier_sub <- gtex_phenoplier %>% 
dplyr::filter(fdr < 0.05) %>% 
dplyr::select(phenotype_desc, lv, fdr) %>% 
dplyr::rename(FDR = fdr) %>% 
dplyr::rename(LV = lv) %>%
dplyr::arrange(FDR)


head(gtex_phenoplier_sub)

,phenotype_desc,LV,FDR
,<fct>,<fct>,<dbl>
278512,Reticulocyte percentage,LV52,1.764323e-31
637364,Monocyte percentage,LV250,8.767880e-31
1402860,Reticulocyte count,LV52,7.587582e-28
1014344,"Non-cancer illness code, self-reported: malabsorption/coeliac disease",LV324,1.316907e-27
1475372,Immature reticulocyte fraction,LV52,2.916401e-25
789392,"Non-cancer illness code, self-reported: sarcoidosis",LV324,3.299579e-24


## Tissue annotations

In [50]:
gtex_meta <- read.table(
  here("data/gtex/GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt"),
  sep = "\t",
  header = TRUE,
  quote = "",
  fill = TRUE,
  comment.char = ""
)

head(gtex_meta)

,SAMPID,SMATSSCR,SMCENTER,SMPTHNTS,SMRIN,SMTS,SMTSD,SMUBRID,SMTSISCH,SMTSPAX,⋯,SME1ANTI,SMSPLTRD,SMBSMMRT,SME1SNSE,SME1PCTS,SMRRNART,SME1MPRT,SMNUM5CD,SMDPMPRT,SME2PCTS
,<chr>,<int>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<int>,<int>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<int>,<dbl>
1,GTEX-1117F-0003-SM-58Q7G,NA,B1,,NA,Blood,Whole Blood,0013756,1188,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,GTEX-1117F-0003-SM-5DWSB,NA,B1,,NA,Blood,Whole Blood,0013756,1188,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
3,GTEX-1117F-0003-SM-6WBT7,NA,B1,,NA,Blood,Whole Blood,0013756,1188,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
4,GTEX-1117F-0011-R10a-SM-AHZ7F,NA,"B1, A1",,NA,Brain,Brain - Frontal Cortex (BA9),0009834,1193,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
5,GTEX-1117F-0011-R10b-SM-CYKQ8,NA,"B1, A1",,7.2,Brain,Brain - Frontal Cortex (BA9),0009834,1193,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
6,GTEX-1117F-0226-SM-5GZZ7,0,B1,"2 pieces, ~15% vessel stroma, rep delineated",6.8,Adipose Tissue,Adipose - Subcutaneous,0002190,1214,1125,⋯,14648800,11999300,0.00315785,14669500,50.0354,0.00310538,0.99474,NA,0,50.1944


## Infer maximum tissue‐specific D‐statistic achieved by any latent variable (LV)

The d-statistic you used is a standardized mean difference: for each tissue and each LV, you compare the LV’s average score in that tissue versus all other tissues (“rest”), and divide that difference by the pooled within-group standard deviation. In practice, it measures how strongly (and consistently) an LV is shifted up (positive d) or down (negative d) in one tissue relative to the rest, expressed in units of within-sample variability—so it behaves like a signal-to-noise effect size, which is why it often ranks tissue-aligned LVs better than pure specificity metrics that ignore variance.

In [51]:
B <- as.matrix(clamp_gtex_data$B)

meta <- gtex_meta[, c("SAMPID", "SMTS")]
meta <- meta[match(colnames(B), meta$SAMPID), , drop = FALSE]
stopifnot(identical(meta$SAMPID, colnames(B)))

tissues <- unique(meta$SMTS)

d_all <- do.call(rbind, lapply(tissues, function(tis) {
  idx <- meta$SMTS == tis
  n1 <- sum(idx); n0 <- sum(!idx)

  m1 <- rowMeans(B[, idx, drop = FALSE])
  m0 <- rowMeans(B[, !idx, drop = FALSE])
  v1 <- apply(B[, idx, drop = FALSE], 1, var)
  v0 <- apply(B[, !idx, drop = FALSE], 1, var)

  sp <- sqrt(((n1 - 1) * v1 + (n0 - 1) * v0) / (n1 + n0 - 2))
  d  <- (m1 - m0) / sp

  data.frame(
    tissue = tis,
    LV = rownames(B),
    d = d,
    stringsAsFactors = FALSE
  )
}))

tissue_top_lv <- d_all %>%
  group_by(tissue) %>%
  slice_max(order_by = d, n = 3, with_ties = FALSE) %>%
  ungroup() %>%
  arrange(desc(d))

In [52]:
head(tissue_top_lv)

tissue,LV,d
<chr>,<chr>,<dbl>
Testis,LV6,36.94441
Kidney,LV50,33.40893
Pancreas,LV25,30.97496
Adrenal Gland,LV20,24.13807
Spleen,LV338,23.12563
Ovary,LV266,19.89569


In [53]:
clamp_gtex_summary_sub <- clamp_gtex_data$summary %>% 
dplyr::filter(AUC > 0.7) %>% 
dplyr::filter(FDR < 0.05)

In [54]:
tissue_best_lv_df <- tissue_top_lv %>%
  dplyr::left_join(gtex_phenoplier_sub, by = "LV", relationship = "many-to-many") %>%
  left_join(clamp_gtex_summary_sub, by = "LV", relationship = 'many-to-many') %>% 
  dplyr::arrange(tissue) %>% 
  dplyr::select(tissue, phenotype_desc, pathway)

head(tissue_best_lv_df)

tissue,phenotype_desc,pathway
<chr>,<fct>,<chr>
Adipose Tissue,Malignant neoplasm of breast,NA
Adipose Tissue,Diagnoses - main ICD10: C50 Malignant neoplasm of breast,NA
Adipose Tissue,"Non-cancer illness code, self-reported: gastrointestinal bleeding",NA
Adipose Tissue,NA,NA
Adipose Tissue,Comparative body size at age 10,Endothelial Cell Lung Mouse
Adipose Tissue,"Cancer code, self-reported: tongue cancer",Endothelial Cell Lung Mouse


In [55]:
write.table(tissue_best_lv_df, here("output/gtex/gtex_tissue_phenoplier.csv"), row.names = FALSE, quote = FALSE, sep = ";")

## Fitered traits and pathways per tissue

| tissue         | phenotype_desc                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           | pathway                                                                                                     |
|----------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------------|
| Adipose Tissue | Comparative body size at age 10                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           | NA                                                                                                         |
| Adrenal Gland  | Non-cancer illness code, self-reported: hypertension; Treatment/medication code: bendroflumethiazide (20003_1141194794); Medication for cholesterol, blood pressure or diabetes: Blood pressure medication                                                                                                                                                                                                                                                                                                                                | NA                                                                                                         |
| Blood          | Platelet distribution width; Mean platelet (thrombocyte) volume                                                                                                                                                                                                                                                                                                                                                                                                                                                                          | Megakaryocyte Blood Human                                                                                  |
| Brain          | Number of incorrect matches in round; Time to complete round                                                                                                                                                                                                                                                                                                                                                                                                                                                                             | Excitatory Neuron Brain Human                                                                              |
| Heart          | Pulse rate; Pulse rate, automated reading; QRS duration                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  | Cardiomyocyte Heart Human                                                                                  |
| Kidney         | Non-cancer illness code, self-reported: gout; Treatment/medication code: allopurinol (20003_1140875408)                                                                                                                                                                                                                                                                                                                                                                                                                                 | Nephron Epithelial Cell Kidney Human; Proximal Tubule Cell Kidney Mouse                                     |
| Liver          | Non-cancer illness code, self-reported: high cholesterol; Treatment/medication code: simvastatin (20003_1140861958); Treatment/medication code: atorvastatin (20003_1141146234); Treatment/medication code: ezetimibe (20003_1141192736); Treatment/medication code: lipitor 10mg tablet (20003_1141146138); Medication for cholesterol, blood pressure or diabetes: Cholesterol lowering medication; Medication for cholesterol, blood pressure, diabetes, or take exogenous hormones: Cholesterol lowering medication; Non-cancer illness code, self-reported: cholelithiasis/gall stones; Diagnoses - main ICD10: K80 Cholelithiasis; Disorders of gallbladder, biliary tract and pancreas | Hepatocyte Liver Mouse                                                                                      |
| Lung           | Peak expiratory flow (PEF)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               | NA                                                                                                         |
| Pancreas       | Diagnoses - main ICD10: C25 Malignant neoplasm of pancreas; Disorders of gallbladder, biliary tract and pancreas                                                                                                                                                                                                                                                                                                                                                                                                                         | Acinar Cell Pancreatic Islet Human                                                                         |
| Prostate       | Malignant neoplasm of prostate; Diagnoses - main ICD10: C61 Malignant neoplasm of prostate; malignant neoplasm of male genital organs; C_MALE_GENITAL                                                                                                                                                                                                                                                                                                                                                                                    | NA                                                                                                         |
| Skin           | Non-cancer illness code, self-reported: eczema/dermatitis; Non-cancer illness code, self-reported: psoriasis; Treatment/medication code: dovobet ointment (20003_1141179992)                                                                                                                                                                                                                                                                                                                                                           | NA                                                                                                         |
| Cervix Uteri   | Diagnoses - main ICD10: N81 Female genital prolapse                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       | NA                                                                                                         |
| Vagina         | Diagnoses - main ICD10: N81 Female genital prolapse                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       | NA                                                                                                         |


In [60]:
B <- as.matrix(clamp_gtex_data$B)

meta <- gtex_meta[, c("SAMPID", "SMTSD")]
meta <- meta[match(colnames(B), meta$SAMPID), , drop = FALSE]
stopifnot(identical(meta$SAMPID, colnames(B)))

tissues <- unique(meta$SMTS)

d_all <- do.call(rbind, lapply(tissues, function(tis) {
  idx <- meta$SMTS == tis
  n1 <- sum(idx); n0 <- sum(!idx)

  m1 <- rowMeans(B[, idx, drop = FALSE])
  m0 <- rowMeans(B[, !idx, drop = FALSE])
  v1 <- apply(B[, idx, drop = FALSE], 1, var)
  v0 <- apply(B[, !idx, drop = FALSE], 1, var)

  sp <- sqrt(((n1 - 1) * v1 + (n0 - 1) * v0) / (n1 + n0 - 2))
  d  <- (m1 - m0) / sp

  data.frame(
    tissue = tis,
    LV = rownames(B),
    d = d,
    stringsAsFactors = FALSE
  )
}))

tissue_top_lv <- d_all %>%
  group_by(tissue) %>%
  slice_max(order_by = d, n = 3, with_ties = FALSE) %>%
  ungroup() %>%
  arrange(desc(d))

In [61]:
tissue_best_lv_df <- tissue_top_lv %>%
  dplyr::left_join(gtex_phenoplier_sub, by = "LV", relationship = "many-to-many") %>%
  left_join(clamp_gtex_summary_sub, by = "LV", relationship = 'many-to-many') %>% 
  dplyr::arrange(tissue) %>% 
  dplyr::select(tissue, phenotype_desc, pathway)

head(tissue_best_lv_df)

tissue,phenotype_desc,pathway
<chr>,<fct>,<chr>
Adipose - Subcutaneous,NA,NA
Adipose - Subcutaneous,Comparative body size at age 10,Endothelial Cell Lung Mouse
Adipose - Subcutaneous,"Cancer code, self-reported: tongue cancer",Endothelial Cell Lung Mouse
Adipose - Subcutaneous,Facial ageing,Fibroblast Skin Mouse
Adipose - Subcutaneous,Facial ageing,Prehypertrophic Chondrocyte Articular Cartilage Human
Adipose - Visceral (Omentum),NA,Wnt2+ Cell Lung Mouse


In [62]:
write.table(tissue_best_lv_df, here("output/gtex/gtex_tissue_phenoplier_SMTSD.csv"), row.names = FALSE, quote = FALSE, sep = ";")

| tissue                                   | phenotype_desc                                                                                                                                                                                                                                                     | pathway                                                          |
|------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------------------------------------------------------------------|
| Artery - Coronary                        | Ischaemic heart disease, wide definition \| Major coronary heart disease event \| Major coronary heart disease event excluding revascularizations                                                                                                                  | NA                                                               |
| Heart - Atrial Appendage                 | Pulse rate \| Pulse rate, automated reading \| QRS duration                                                                                                                                                                                                         | Cardiomyocyte Heart Human                                        |
| Heart - Left Ventricle                   | Pulse rate \| Pulse rate, automated reading \| QRS duration                                                                                                                                                                                                         | Cardiomyocyte Heart Human                                        |
| Brain - Amygdala                         | Number of incorrect matches in round \| Time to complete round                                                                                                                                                                                                      | Excitatory Neuron Brain Human                                    |
| Brain - Anterior cingulate cortex (BA24) | Number of incorrect matches in round \| Time to complete round                                                                                                                                                                                                      | Excitatory Neuron Brain Human                                    |
| Brain - Cortex                           | Number of incorrect matches in round \| Time to complete round                                                                                                                                                                                                      | Excitatory Neuron Brain Human                                    |
| Brain - Frontal Cortex (BA9)             | Number of incorrect matches in round \| Time to complete round                                                                                                                                                                                                      | Excitatory Neuron Brain Human                                    |
| Kidney - Cortex                          | Non-cancer illness code, self-reported: gout \| Treatment/medication code: allopurinol (20003_1140875408)                                                                                                                                                          | Nephron Epithelial Cell Kidney Human \| Proximal Tubule Cell Kidney Mouse |
| Kidney - Medulla                         | Non-cancer illness code, self-reported: gout \| Treatment/medication code: allopurinol (20003_1140875408)                                                                                                                                                          | Nephron Epithelial Cell Kidney Human \| Proximal Tubule Cell Kidney Mouse |
| Liver                                    | Non-cancer illness code, self-reported: high cholesterol \| Treatment/medication code: simvastatin (20003_1140861958) \| Treatment/medication code: atorvastatin (20003_1141146234) \| Treatment/medication code: ezetimibe (20003_1141192736) \| Treatment/medication code: lipitor 10mg tablet (20003_1141146138) \| Medication for cholesterol, blood pressure or diabetes: Cholesterol lowering medication \| Diagnoses - main ICD10: K80 Cholelithiasis \| Disorders of gallbladder, biliary tract and pancreas | Hepatocyte Liver Mouse                                           |
| Lung                                     | Peak expiratory flow (PEF)                                                                                                                                                                                                                                          | NA                                                               |
| Pancreas                                 | Diagnoses - main ICD10: C25 Malignant neoplasm of pancreas \| Disorders of gallbladder, biliary tract and pancreas                                                                                                                                                 | Acinar Cell Pancreatic Islet Human                               |
| Prostate                                 | Malignant neoplasm of prostate \| Diagnoses - main ICD10: C61 Malignant neoplasm of prostate \| malignant neoplasm of male genital organs \| C_MALE_GENITAL                                                                                                       | NA                                                               |
| Skin - Sun Exposed (Lower leg)           | Non-cancer illness code, self-reported: eczema/dermatitis \| Non-cancer illness code, self-reported: psoriasis \| Treatment/medication code: dovobet ointment (20003_1141179992)                                                                                | NA                                                               |
| Cervix - Ectocervix                      | Diagnoses - main ICD10: N81 Female genital prolapse                                                                                                                                                                                                                 | NA                                                               |
| Cervix - Endocervix                      | Diagnoses - main ICD10: N81 Female genital prolapse                                                                                                                                                                                                                 | NA                                                               |
| Vagina                                   | Diagnoses - main ICD10: N81 Female genital prolapse                                                                                                                                                                                                                 | NA                                                               |
| Whole Blood                              | Platelet distribution width \| Mean platelet (thrombocyte) volume                                                                                                                                                                                                   | Megakaryocyte Blood Human                                        |
